# 🧠💭 Build a Reasoning Model from Scratch (PyTorch only)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/loki52501/My_own_llm/blob/main/llm-from-scratch/Reasoning_Model_From_Scratch.ipynb)

In `LLM_From_Scratch.ipynb` we built a GPT that learns to **continue text**. This notebook turns that same kind of model into a small **reasoning model**: one that *thinks* first, inside `<think> … </think>`, and then answers inside `<answer> … </answer>`.

It follows the same recipe used for OpenAI o1 / DeepSeek-R1, shrunk down to a model that trains in about 20–30 minutes on a free Colab T4. Everything uses **only PyTorch**, trained on **the same books** (`txt-files.tar`).

```
 ┌────────────────┐   ┌──────────────────┐   ┌─────────────────────┐   ┌────────────────────┐   ┌──────────────────┐
 │ 1. PRETRAIN    │──▶│ 2. REASONING     │──▶│ 3. SFT              │──▶│ 4. RL with GRPO    │──▶│ 5. TEST-TIME     │
 │ on the books   │   │ DATA from books  │   │ "cold start":       │   │ reward = is the    │   │ COMPUTE          │
 │ (next token)   │   │ (verifiable Q&A) │   │ imitate worked      │   │ answer correct?    │   │ sample many,     │
 │                │   │                  │   │ solutions           │   │ (no labels needed  │   │ majority vote    │
 │ learns English │   │ names + objects  │   │ learns the <think>  │   │  for the steps)    │   │                  │
 └────────────────┘   └──────────────────┘   └─────────────────────┘   └────────────────────┘   └──────────────────┘
```

### What makes a model a "reasoning model"?

| | Normal LLM | Reasoning LLM |
|---|---|---|
| Output | answer straight away | `<think>` step-by-step scratch work `</think>` then `<answer>` |
| Why it helps | each token gets a *fixed* amount of compute | writing intermediate steps = **more compute per question**; each step is an easy sub-problem |
| How it's trained | imitate text | imitate reasoning (SFT) **then** reinforcement learning on problems whose answer can be **checked automatically** |

### Our problems

A tiny 10M-parameter model can't solve maths olympiad questions, but it *can* learn the key behaviour: decomposing a problem into steps. We generate two kinds of **verifiable** problems, using **character names, objects and words taken from your books**:

1. **Story arithmetic.** *"Elizabeth has 7 letters. Darcy gives Elizabeth 5 more letters. Elizabeth loses 3 letters. How many letters does Elizabeth have now?"*
2. **Letter counting.** *"How many times does the letter 'e' appear in 'netherfield'?"* This is famously hard for LLMs because a BPE tokenizer hides individual letters. Thinking lets the model spell the word out first.

We'll test two claims ourselves:

* **"Thinking helps."** We compare the same model answering **directly** against answering **with thinking**.
* **"RL teaches what imitation didn't."** SFT only ever shows stories with 1–3 steps. The **4–5-step** stories are never given a worked solution; the model only gets to *practise* them during RL, where the only feedback is whether its final answer was right.

> **Colab:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. On CPU the notebook runs a tiny version end-to-end, which shows the mechanics but gives weak results.

## 0. Setup

In [ ]:
import os, re, sys, math, time, json, copy, heapq, random, tarfile, urllib.request, collections
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
SEED = 1337
random.seed(SEED); torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
FAST = device == "cuda"          # full-size run on GPU, tiny demo run on CPU
print(f"PyTorch {torch.__version__} | device = {device} | Colab = {IN_COLAB} | full-size run = {FAST}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Load the books 📚

This is the same loader as the first notebook. It looks for `txt-files.tar` in this order:

1. the working directory, or `/content` (drag the file into Colab's file panel),
2. Google Drive at `MyDrive/llm_from_scratch/txt-files.tar` (set `USE_GOOGLE_DRIVE = True`),
3. the GitHub repo at `llm-from-scratch/data/txt-files.tar`,
4. an upload button,
5. and if all of those fail, a few public-domain Gutenberg books.

In [ ]:
USE_GOOGLE_DRIVE  = False
UPLOAD_IF_MISSING = True
TAR_NAME          = "txt-files.tar"
GITHUB_TAR_URL    = "https://raw.githubusercontent.com/loki52501/My_own_llm/main/llm-from-scratch/data/txt-files.tar"

WORK_DIR  = Path("reasoning_work"); WORK_DIR.mkdir(exist_ok=True)
BOOKS_DIR = WORK_DIR / "books";     BOOKS_DIR.mkdir(exist_ok=True)

if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")

candidates = [Path(TAR_NAME), Path("data") / TAR_NAME, Path("llm-from-scratch/data") / TAR_NAME,
              Path("/content") / TAR_NAME, Path("/content/drive/MyDrive/llm_from_scratch") / TAR_NAME,
              Path("/content/drive/MyDrive") / TAR_NAME]
tar_path = next((p for p in candidates if p.exists()), None)

if tar_path is None:
    try:
        urllib.request.urlretrieve(GITHUB_TAR_URL, TAR_NAME); tar_path = Path(TAR_NAME)
    except Exception as e:
        print("not on GitHub:", e)

if tar_path is None and IN_COLAB and UPLOAD_IF_MISSING:
    from google.colab import files
    print(f"Upload {TAR_NAME} (or .txt files):")
    for name, blob in files.upload().items():
        dest = Path(name) if name.endswith(".tar") else BOOKS_DIR / name
        dest.write_bytes(blob)
        if name.endswith(".tar"): tar_path = dest

if tar_path is not None:
    with tarfile.open(tar_path) as tar:
        for m in tar.getmembers():
            if m.isfile() and m.name.lower().endswith(".txt"):
                with tar.extractfile(m) as src:
                    (BOOKS_DIR / Path(m.name).name).write_bytes(src.read())

if not list(BOOKS_DIR.glob("*.txt")):
    print("No books found, downloading from Project Gutenberg ...")
    for name, num in [("pride_and_prejudice", 1342), ("sherlock_holmes", 1661), ("alice", 11),
                      ("frankenstein", 84), ("moby_dick", 2701)]:
        try: urllib.request.urlretrieve(f"https://www.gutenberg.org/cache/epub/{num}/pg{num}.txt", BOOKS_DIR / f"{name}.txt")
        except Exception as e: print("  ✗", name, e)

EOT = "<|endoftext|>"
def clean(t):
    t = t.replace("\r\n", "\n").replace("\r", "\n").replace("\ufeff", "")
    # drop the Project Gutenberg header/footer (licence text) if present
    start = re.search(r"\*\*\* ?START OF (?:THE|THIS) PROJECT GUTENBERG.*?\*\*\*", t)
    end = re.search(r"\*\*\* ?END OF (?:THE|THIS) PROJECT GUTENBERG", t)
    if start: t = t[start.end():end.start() if end else None]
    return re.sub(r"\n{3,}", "\n\n", t).strip()

book_files = sorted(BOOKS_DIR.glob("*.txt"))
raw_text = f"\n{EOT}\n".join(clean(f.read_text(encoding="utf-8", errors="ignore")) for f in book_files)
print(f"{len(book_files)} books, {len(raw_text):,} characters")

## 2. Mine the books for names, objects and words 🔎

Reasoning problems are more fun, and the model reads them more easily, when they use the vocabulary it saw during pretraining. We extract three lists with simple regex tricks:

| List | Trick | Example |
|---|---|---|
| **Names** | Capitalized words next to *said / cried / replied* ("…," said **Elizabeth**) are characters | `Elizabeth`, `Darcy`, `Holmes` |
| **Objects** | Plural words that follow a number word ("two **letters**", "three **horses**") are countable nouns | `letters`, `horses`, `books` |
| **Words** | The 400 most frequent lowercase words of 4–8 letters, for letter-counting | `little`, `creature` |

In [ ]:
speech = r"(?:said|cried|replied|asked|answered|exclaimed|continued)"
speakers = collections.Counter(re.findall(rf"\b{speech} (?:Mr\. |Mrs\. |Miss )?([A-Z][a-z]{{2,11}})\b", raw_text) +
                               re.findall(rf"\b([A-Z][a-z]{{2,11}}) {speech}\b", raw_text))
lower_words = set(re.findall(r"\b[a-z]+\b", raw_text))
TITLES = {"Mr", "Mrs", "Miss", "Sir", "Lady", "Lord", "Dr", "Madam", "The", "And", "But", "Project", "Gutenberg"}
NAMES = [w for w, c in speakers.most_common(200) if w.lower() not in lower_words and c >= 2 and w not in TITLES][:50]

number_words = r"(?:two|three|four|five|six|seven|eight|nine|ten|twenty|several|many|few)"
plurals = collections.Counter(re.findall(rf"\b{number_words} ([a-z]{{3,10}}s)\b", raw_text))
TIME_WORDS = {"days", "years", "weeks", "months", "hours", "minutes", "times", "others", "miles", "yards",
              "feet", "seconds", "moments", "nights", "things", "ones", "persons", "people"}
OBJECTS = [w for w, c in plurals.most_common(200) if w not in TIME_WORDS and not w.endswith("ss")][:40]

word_counts = collections.Counter(re.findall(r"\b[a-z]{4,8}\b", raw_text))
WORDS = [w for w, c in word_counts.most_common(400)]

# fallbacks for a very small / unusual corpus
if len(NAMES) < 10:   NAMES += ["Alice", "Elizabeth", "Darcy", "Holmes", "Watson", "Victor", "Jane", "Ahab", "Emma", "Tom"]
if len(OBJECTS) < 10: OBJECTS += ["letters", "books", "apples", "horses", "coins", "candles", "rings", "flowers", "cups", "keys"]

print("NAMES  :", NAMES[:20])
print("OBJECTS:", OBJECTS[:20])
print("WORDS  :", WORDS[100:120])
print(f"{len(NAMES)} names, {len(OBJECTS)} objects, {len(WORDS)} words")

## 3. Generate verifiable reasoning problems 🧩

Each generator returns a **question**, a worked **chain of thought**, and the **answer**. The answer can be checked by a program, which is what makes reinforcement learning possible later: we don't need a human to grade anything.

### The chat / reasoning format

We add special tokens that the tokenizer will never split:

```
<|user|>Think step by step. Elizabeth has 7 letters. Darcy gives Elizabeth 5 more letters. How many letters does Elizabeth have now?
<|assistant|><think>
Elizabeth starts with 7.
7 + 5 = 12
</think><answer>12</answer><|endoftext|>
```

There are two prompt modes, so we can later measure whether thinking helps:

| Mode | Instruction | Model must output |
|---|---|---|
| **think** | `Think step by step.` | `<think> … </think><answer>X</answer>` |
| **direct** | `Answer directly.` | `<answer>X</answer>` with no scratch work |

In [ ]:
def make_arithmetic(rng, min_ops=1, max_ops=3):
    """A short story with min_ops..max_ops operations. Returns (question, chain_of_thought, answer)."""
    a, b = rng.sample(NAMES, 2)
    obj = rng.choice(OBJECTS)
    x = rng.randint(2, 15)                       # small numbers keep the arithmetic learnable
    story = [f"{a} has {x} {obj}."]
    steps = [f"{a} starts with {x}."]
    for _ in range(rng.randint(min_ops, max_ops)):
        op = rng.choice(["get", "find", "give", "lose", "double"])
        if op == "double" and x > 20: op = "lose"
        if op in ("give", "lose") and x < 2: op = "find"
        if op in ("get", "find") and x > 40: op = "lose"
        if op == "get":
            k = rng.randint(1, 9); story.append(f"{b} gives {a} {k} more {obj}.")
            steps.append(f"{x} + {k} = {x + k}"); x += k
        elif op == "find":
            k = rng.randint(1, 9); story.append(f"{a} finds {k} {obj}.")
            steps.append(f"{x} + {k} = {x + k}"); x += k
        elif op == "give":
            k = rng.randint(1, x); story.append(f"{a} gives {k} {obj} to {b}.")
            steps.append(f"{x} - {k} = {x - k}"); x -= k
        elif op == "lose":
            k = rng.randint(1, x); story.append(f"{a} loses {k} {obj}.")
            steps.append(f"{x} - {k} = {x - k}"); x -= k
        else:
            story.append(f"Then the number of {obj} {a} has doubles.")
            steps.append(f"{x} * 2 = {2 * x}"); x *= 2
    question = " ".join(story) + f" How many {obj} does {a} have now?"
    return question, "\n".join(steps), str(x)

def make_letter_count(rng):
    """Count a letter in a word from the books. The CoT spells the word with a running counter."""
    word = rng.choice(WORDS)
    letter = rng.choice(word) if rng.random() < 0.75 else rng.choice("abcdefghijklmnopqrstuvwxyz")
    seen, spelled = 0, []
    for ch in word:
        if ch == letter:
            seen += 1; spelled.append(f"{ch}={seen}")
        else:
            spelled.append(ch)
    question = f"How many times does the letter '{letter}' appear in '{word}'?"
    cot = "Spell it: " + " ".join(word) + f"\nCount '{letter}': " + ", ".join(spelled) + f"\nTotal: {seen}"
    return question, cot, str(seen)

TASKS = {"arithmetic": make_arithmetic, "letters": make_letter_count}           # used for SFT
HARD_TASKS = {"arith_4-5_steps": lambda rng: make_arithmetic(rng, 4, 5)}          # NEVER shown in SFT
ALL_TASKS = {**TASKS, **HARD_TASKS}

USER, ASSISTANT = "<|user|>", "<|assistant|>"
THINK_OPEN, THINK_CLOSE, ANS_OPEN, ANS_CLOSE = "<think>", "</think>", "<answer>", "</answer>"
SPECIAL_TOKENS = [EOT, USER, ASSISTANT, THINK_OPEN, THINK_CLOSE, ANS_OPEN, ANS_CLOSE]
MODE_TEXT = {"think": "Think step by step.", "direct": "Answer directly."}

def make_problem(rng, task=None):
    task = task or rng.choice(list(TASKS))       # default: only the SFT tasks
    q, cot, ans = ALL_TASKS[task](rng)
    return {"task": task, "question": q, "cot": cot, "answer": ans}

def prompt_text(p, mode="think"):
    return f"{USER}{MODE_TEXT[mode]} {p['question']}\n{ASSISTANT}"

def target_text(p, mode="think"):
    if mode == "think":
        return f"{THINK_OPEN}\n{p['cot']}\n{THINK_CLOSE}{ANS_OPEN}{p['answer']}{ANS_CLOSE}{EOT}"
    return f"{ANS_OPEN}{p['answer']}{ANS_CLOSE}{EOT}"

rng = random.Random(0)
for task in TASKS:
    p = make_problem(rng, task)
    print("━" * 90)
    print(prompt_text(p, "think") + target_text(p, "think"))

## 4. Tokenizer with reasoning tokens ✂️

This is the same **byte-level BPE** as the first notebook, written from scratch, with two changes:

* The special tokens `<|user|> <|assistant|> <think> </think> <answer> </answer> <|endoftext|>` each get **one id**. The model can then produce "start thinking" or "give the answer" as a single decision.
* We train the merges on the books **plus a sample of problem text**, so numbers like `" 12"` become single tokens (numbers are rare in novels).

```
 "<answer>12</answer>"  ──▶  [ <answer> ][ 12 ][ </answer> ]   (3 tokens, not 19 bytes)
```

In [ ]:
PRETOKENIZE = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?[^\W\d_]+| ?\d{1,3}| ?[^\s\w]+| ?_+|\s+(?!\S)|\s+""")

def merge_ids(ids, pair, new_id):
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(new_id); i += 2
        else:
            out.append(ids[i]); i += 1
    return out

class BPETokenizer:
    def __init__(self, special_tokens):
        self.merges, self.vocab = {}, {i: bytes([i]) for i in range(256)}
        self.special_tokens, self.special, self._cache = list(special_tokens), {}, {}

    def train(self, text, vocab_size):
        for sp in self.special_tokens:
            text = text.replace(sp, " ")
        word_freq = collections.Counter(PRETOKENIZE.findall(text))
        words = [list(w.encode("utf-8")) for w in word_freq]
        freqs = list(word_freq.values())
        pair_counts, where = collections.defaultdict(int), collections.defaultdict(set)
        for wi, (ids, f) in enumerate(zip(words, freqs)):
            for p in zip(ids, ids[1:]):
                pair_counts[p] += f; where[p].add(wi)
        heap = [(-c, p) for p, c in pair_counts.items()]; heapq.heapify(heap)
        for i in range(vocab_size - 256 - len(self.special_tokens)):
            while heap:
                neg_c, pair = heapq.heappop(heap)
                if pair_counts.get(pair, 0) == -neg_c and -neg_c > 0: break
            else:
                break
            new_id = 256 + i
            self.merges[pair] = new_id
            self.vocab[new_id] = self.vocab[pair[0]] + self.vocab[pair[1]]
            changed = set()
            for wi in list(where[pair]):
                old = words[wi]; new = merge_ids(old, pair, new_id)
                if new == old: continue
                f = freqs[wi]
                for p in zip(old, old[1:]): pair_counts[p] -= f; changed.add(p)
                for p in zip(new, new[1:]): pair_counts[p] += f; where[p].add(wi); changed.add(p)
                words[wi] = new
            for p in changed:
                if pair_counts[p] > 0: heapq.heappush(heap, (-pair_counts[p], p))
            del pair_counts[pair], where[pair]
        for sp in self.special_tokens:
            self.special[sp] = len(self.vocab); self.vocab[len(self.vocab)] = sp.encode("utf-8")
        self._split = re.compile("(" + "|".join(re.escape(s) for s in self.special) + ")")
        return self

    @property
    def vocab_size(self): return len(self.vocab)

    def _encode_chunk(self, chunk):
        if chunk not in self._cache:
            ids = list(chunk.encode("utf-8"))
            while len(ids) >= 2:
                pair = min(zip(ids, ids[1:]), key=lambda p: self.merges.get(p, float("inf")))
                if pair not in self.merges: break
                ids = merge_ids(ids, pair, self.merges[pair])
            self._cache[chunk] = ids
        return self._cache[chunk]

    def encode(self, text):
        ids = []
        for part in self._split.split(text):
            if part in self.special: ids.append(self.special[part])
            elif part:
                for chunk in PRETOKENIZE.findall(part): ids.extend(self._encode_chunk(chunk))
        return ids

    def decode(self, ids):
        return b"".join(self.vocab[i] for i in ids).decode("utf-8", errors="replace")

    def save(self, path):
        json.dump({"merges": [[a, b, n] for (a, b), n in self.merges.items()], "special": self.special}, open(path, "w"))

In [ ]:
VOCAB_SIZE = 4096
rng = random.Random(1)
problem_sample = "\n".join(prompt_text(p) + target_text(p) for p in (make_problem(rng) for _ in range(20_000)))

t0 = time.time()
tok = BPETokenizer(SPECIAL_TOKENS).train(raw_text[:15_000_000] + "\n" + problem_sample, VOCAB_SIZE)
tok.save(WORK_DIR / "tokenizer.json")
print(f"vocab {tok.vocab_size} trained in {time.time()-t0:.1f}s")

EOT_ID, THINK_CLOSE_ID, ANS_CLOSE_ID = tok.special[EOT], tok.special[THINK_CLOSE], tok.special[ANS_CLOSE]

p = make_problem(random.Random(3), "arithmetic")
ids = tok.encode(prompt_text(p) + target_text(p))
print(f"\n{len(ids)} tokens:")
print("|".join(tok.decode([i]) for i in ids))
assert tok.decode(ids) == prompt_text(p) + target_text(p)

## 5. The GPT model 🏗️

This is the same architecture as the first notebook (see there for the step-by-step build of attention). Here it is compacted into one cell:

```
 ids ─▶ token emb + pos emb ─▶ [ LayerNorm → causal multi-head attention → + ─▶ LayerNorm → MLP → + ] × N ─▶ LayerNorm ─▶ logits
```

Nothing about the *architecture* changes for reasoning models. **Reasoning comes entirely from the data and the training method.**

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.n_heads, self.dropout = n_heads, dropout
        self.qkv, self.proj = nn.Linear(d, 3 * d, bias=False), nn.Linear(d, d)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = (t.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2) for t in self.qkv(x).split(C, 2))
        # softmax(Q·Kᵀ/√d + causal mask)·V   (fused PyTorch kernel, same math as notebook 1)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        return self.resid_drop(self.proj(y.transpose(1, 2).contiguous().view(B, T, C)))

class Block(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = CausalSelfAttention(d, n_heads, dropout)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(approximate="tanh"), nn.Linear(4 * d, d), nn.Dropout(dropout))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["d_model"])
        self.drop = nn.Dropout(cfg["dropout"])
        self.blocks = nn.ModuleList([Block(cfg["d_model"], cfg["n_heads"], cfg["dropout"]) for _ in range(cfg["n_layers"])])
        self.ln_f = nn.LayerNorm(cfg["d_model"])
        self.head = nn.Linear(cfg["d_model"], cfg["vocab_size"], bias=False)
        self.head.weight = self.tok_emb.weight                       # weight tying
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, 0.0, 0.02)
                if getattr(m, "bias", None) is not None: nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        T = idx.shape[1]
        x = self.drop(self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device)))
        for blk in self.blocks: x = blk(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:     # ignore_index=-100 lets us mask out tokens we don't want to learn
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss

if FAST:
    cfg = dict(vocab_size=tok.vocab_size, context_length=256, d_model=384, n_heads=6, n_layers=6, dropout=0.1)
else:
    cfg = dict(vocab_size=tok.vocab_size, context_length=192, d_model=128, n_heads=4, n_layers=4, dropout=0.0)

torch.manual_seed(SEED)
model = GPT(cfg).to(device)
print(json.dumps(cfg), f"\nparameters: {sum(p.numel() for p in model.parameters())/1e6:.2f} M")

use_amp = device == "cuda"
amp_dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16
def autocast():
    return torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp)

## 6. Stage 1: Pretraining on the books 📖

The model first learns **language**: grammar, names and common words, by predicting the next token of the books (exactly as in notebook 1). A pretrained base is what makes the later stages work. The model already "knows" what `Elizabeth` or `letters` are, and it only has to learn the *task*.

```
 books: "...Elizabeth took up the letter and read it..."
 x:      Elizabeth  took   up    the   letter
 y:      took       up     the   letter  and      ← predict the next token everywhere
```

In [ ]:
t0 = time.time()
book_ids = torch.tensor(tok.encode(raw_text), dtype=torch.long)
n = int(0.95 * len(book_ids))
book_train, book_val = book_ids[:n], book_ids[n:]
print(f"{len(book_ids):,} book tokens (encoded in {time.time()-t0:.1f}s)")

def book_batch(data, B, T):
    ix = torch.randint(len(data) - T - 1, (B,))
    x = torch.stack([data[i:i + T] for i in ix]); y = torch.stack([data[i + 1:i + T + 1] for i in ix])
    return x.to(device), y.to(device)

def lr_schedule(step, max_steps, max_lr, warmup):
    if step < warmup: return max_lr * (step + 1) / warmup
    prog = (step - warmup) / max(1, max_steps - warmup)
    return max_lr * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(prog, 1.0))))

def make_optimizer(model, lr, wd=0.1):
    decay = [p for p in model.parameters() if p.dim() >= 2]
    no_decay = [p for p in model.parameters() if p.dim() < 2]
    return torch.optim.AdamW([{"params": decay, "weight_decay": wd}, {"params": no_decay, "weight_decay": 0.0}],
                             lr=lr, betas=(0.9, 0.95))

def train_steps(model, get_batch, max_steps, max_lr, warmup, log_every, name):
    opt = make_optimizer(model, max_lr)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and amp_dtype == torch.float16)
    losses, t0 = [], time.time()
    model.train()
    for step in range(max_steps):
        for g in opt.param_groups: g["lr"] = lr_schedule(step, max_steps, max_lr, warmup)
        x, y = get_batch()
        with autocast():
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        losses.append(loss.item())
        if step % log_every == 0 or step == max_steps - 1:
            print(f"[{name}] step {step:5d}/{max_steps} | loss {sum(losses[-log_every:])/len(losses[-log_every:]):.3f} | {time.time()-t0:5.0f}s")
    return losses

PRETRAIN_STEPS = 2500 if FAST else 300
BATCH = 64 if FAST else 16
T_PRE = cfg["context_length"]
pretrain_losses = train_steps(model, lambda: book_batch(book_train, BATCH, T_PRE), PRETRAIN_STEPS,
                              max_lr=6e-4 if FAST else 1e-3, warmup=100 if FAST else 30,
                              log_every=250 if FAST else 50, name="pretrain")

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, top_k=None, stop_ids=(EOT_ID,)):
    """Batched sampling. temperature=0 → greedy. Stops early once every row has emitted a stop token."""
    model.eval()
    done = torch.zeros(idx.shape[0], dtype=torch.bool, device=idx.device)
    for _ in range(max_new_tokens):
        with autocast():
            logits, _ = model(idx[:, -model.cfg["context_length"]:])
        logits = logits[:, -1, :].float()
        if temperature == 0:
            nxt = logits.argmax(-1, keepdim=True)
        else:
            logits = logits / temperature
            if top_k:
                v, _ = torch.topk(logits, top_k); logits[logits < v[:, [-1]]] = -float("inf")
            nxt = torch.multinomial(torch.softmax(logits, -1), 1)
        nxt[done] = EOT_ID                          # finished rows just pad with EOT
        idx = torch.cat([idx, nxt], 1)
        done |= torch.isin(nxt.squeeze(1), torch.tensor(stop_ids, device=idx.device))
        if done.all(): break
    return idx

def continue_text(prompt, n=60, temperature=0.8):
    x = torch.tensor([tok.encode(prompt)], device=device)
    return tok.decode(generate(model, x, n, temperature, top_k=50)[0].tolist())

plt.figure(figsize=(6, 3)); plt.plot(pretrain_losses, alpha=.5)
plt.title("Stage 1: pretraining loss"); plt.xlabel("step"); plt.ylabel("loss"); plt.grid(alpha=.3); plt.show()
print(continue_text(f"{NAMES[0]} said"))

### Can the *base* model already reason?

Let's ask it a question in our format before any fine-tuning. It has never seen `<think>` or `<answer>`, so it will just ramble on in book-style English. **The format and the skill both have to be taught.**

In [ ]:
p = make_problem(random.Random(7), "arithmetic")
x = torch.tensor([tok.encode(prompt_text(p))], device=device)
print(tok.decode(generate(model, x, 60, temperature=0.8, top_k=50)[0].tolist()))
print("\ncorrect answer:", p["answer"])
torch.save(model.state_dict(), WORK_DIR / "base.pt")

## 7. Stage 2: Supervised fine-tuning (SFT) on worked solutions 📝

DeepSeek-R1 calls this the **"cold start"**: show the model a few thousand examples of *good* reasoning so it learns the format and the basic procedure. We train on a 50/50 mix of **think** and **direct** examples.

### Loss masking

We want the model to learn to write the **answer**, not to predict the user's question. So we set the target to `-100` (ignored by `cross_entropy`) on every prompt token and on padding:

```
 tokens :  <|user|> Think step by step . Elizabeth has 7 ... <|assistant|> <think> 7 + 5 = 12 </think> <answer> 12 </answer> <eot> <pad> <pad>
 target :  -100     -100  ...                                -100          ✓       ✓ ✓ ✓ ✓  ✓        ✓        ✓   ✓         ✓     -100  -100
            └──────────────── prompt: no loss ─────────────────────────┘   └────────────── completion: learn this ───────────────┘
```

In [ ]:
def encode_example(p, mode):
    prompt_ids, target_ids = tok.encode(prompt_text(p, mode)), tok.encode(target_text(p, mode))
    ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids
    return ids, labels

def sft_batch(rng, B):
    rows = []
    for _ in range(B):
        mode = "think" if rng.random() < 0.5 else "direct"
        ids, labels = encode_example(make_problem(rng), mode)
        rows.append((ids, labels))
    T = min(max(len(r[0]) for r in rows), cfg["context_length"] + 1)
    x = torch.full((B, T - 1), EOT_ID); y = torch.full((B, T - 1), -100)
    for i, (ids, labels) in enumerate(rows):
        ids, labels = ids[:T], labels[:T]
        x[i, :len(ids) - 1] = torch.tensor(ids[:-1])
        y[i, :len(ids) - 1] = torch.tensor(labels[1:])      # shift by one: predict the next token
    return x.to(device), y.to(device)

xb, yb = sft_batch(random.Random(0), 2)
print("input :", "|".join(tok.decode([i]) for i in xb[0].tolist()))
print("target:", "|".join("·" if i == -100 else tok.decode([i]) for i in yb[0].tolist()))
lengths = [len(encode_example(make_problem(random.Random(i)), "think")[0]) for i in range(500)]
print(f"\nthink-example length: mean {sum(lengths)/len(lengths):.0f}, max {max(lengths)} tokens")

In [ ]:
SFT_STEPS = 2500 if FAST else 400
sft_rng = random.Random(123)
sft_losses = train_steps(model, lambda: sft_batch(sft_rng, BATCH), SFT_STEPS,
                         max_lr=3e-4 if FAST else 1e-3, warmup=100 if FAST else 30,
                         log_every=250 if FAST else 50, name="SFT")
torch.save(model.state_dict(), WORK_DIR / "sft.pt")

plt.figure(figsize=(6, 3)); plt.plot(sft_losses, alpha=.5)
plt.title("Stage 2: SFT loss (completion tokens only)"); plt.xlabel("step"); plt.yscale("log"); plt.grid(alpha=.3); plt.show()

## 8. Evaluation: parse → verify → score ✅

For every test problem we:

1. generate greedily until `<|endoftext|>`,
2. **parse** the text between `<answer>` and `</answer>`,
3. **verify** it against the known answer, with no human needed.

To batch generation without padding tricks, we group problems by prompt length, so every row in a batch has the same length.

In [ ]:
ANS_RE = re.compile(re.escape(ANS_OPEN) + r"(.*?)" + re.escape(ANS_CLOSE), re.S)

def parse_answer(completion):
    m = ANS_RE.search(completion)
    return m.group(1).strip() if m else None

def well_formatted(completion, mode="think"):
    if mode == "think":
        return re.match(r"^<think>.*</think><answer>[^<]*</answer>", completion, re.S) is not None
    return completion.startswith(ANS_OPEN) and parse_answer(completion) is not None

MAX_NEW = 120

@torch.no_grad()
def run_problems(model, problems, mode="think", temperature=0.0, n_samples=1, batch=64):
    """Returns, for every problem, a list of n_samples completion strings."""
    outs = [None] * len(problems)
    groups = collections.defaultdict(list)
    for i, p in enumerate(problems):
        groups[len(tok.encode(prompt_text(p, mode)))].append(i)
    for L, idxs in groups.items():
        per = max(1, batch // n_samples)
        for s in range(0, len(idxs), per):
            chunk = idxs[s:s + per]
            x = torch.tensor([tok.encode(prompt_text(problems[i], mode)) for i in chunk for _ in range(n_samples)], device=device)
            y = generate(model, x, MAX_NEW, temperature=temperature, top_k=None)
            texts = [tok.decode(row[L:].tolist()).split(EOT)[0] for row in y]
            for j, i in enumerate(chunk):
                outs[i] = texts[j * n_samples:(j + 1) * n_samples]
    return outs

def evaluate(model, problems, mode="think"):
    outs = run_problems(model, problems, mode)
    res = collections.defaultdict(list)
    for p, (c,) in zip(problems, outs):
        res[p["task"]].append(parse_answer(c) == p["answer"])
        res["all"].append(parse_answer(c) == p["answer"])
    return {k: sum(v) / len(v) for k, v in res.items()}

eval_rng = random.Random(999)
N_EVAL = 200 if FAST else 60
test_set = [make_problem(eval_rng, t) for t in ALL_TASKS for _ in range(N_EVAL // 2)]

results = {}
for mode in ["direct", "think"]:
    results[f"SFT · {mode}"] = evaluate(model, test_set, mode)
    print(f"SFT, {mode:6s}:", {k: f"{v:.0%}" for k, v in results[f"SFT · {mode}"].items()})

In [ ]:
def show(p, mode="think", temperature=0.0):
    c = run_problems(model, [p], mode, temperature)[0][0]
    ok = parse_answer(c) == p["answer"]
    print(f"Q: {p['question']}\n--- model ({mode}) ---\n{c}\n--- {'✅ correct' if ok else '❌ wrong, expected ' + p['answer']}\n")

show(test_set[0]); show(test_set[N_EVAL // 2]); show(test_set[1], "direct"); show(test_set[-1])

**Does thinking help?** Compare the `direct` and `think` rows above. Direct answering forces the model to do all the arithmetic, or all the letter counting, *inside a single forward pass*. Thinking splits the work into many easy steps, and each step gets a whole forward pass. This is the core reason reasoning models work: **more tokens = more computation.**

## 9. Stage 3: Reinforcement learning with GRPO 🎯

SFT teaches the model to *imitate* solutions. Reinforcement learning lets it **practise**: it tries problems, gets rewarded when the final answer is right, and reinforces whatever reasoning led there. The reward only checks the *answer*, so the model is free to discover its own way of reasoning. This is how DeepSeek-R1-Zero learned to "think longer" and to "double-check".

**GRPO** (Group Relative Policy Optimization, from DeepSeekMath / R1) is PPO without a value network. For each question we sample a **group** of `G` answers and score each one *relative to its group*:

```
                     ┌──▶ completion 1 ──▶ reward 1.0 ──┐
 question ─▶ policy ─┼──▶ completion 2 ──▶ reward 0.0 ──┤   advantage_i = (r_i − mean(r)) / std(r)
   (×G)              ├──▶ completion 3 ──▶ reward 1.0 ──┼──▶  +0.9  −1.1  +0.9  −1.1 ...
                     └──▶ completion G ──▶ reward 0.0 ──┘        │
                                                                 ▼
          loss = − mean over tokens [ advantage · log π(token) ]  +  β · KL(π ‖ π_ref)
                   └─ push UP the tokens of better-than-average answers ─┘   └─ stay close to the SFT model ─┘
```

Key ideas:

* **Relative, not absolute.** If every sample is right (or every one is wrong) the advantages are all 0, so there's nothing to learn from that question. The model learns from questions at the edge of its ability.
* **Verifiable reward** = `1.0` if the parsed answer is correct, plus a small `0.1` bonus for the correct `<think>…</think><answer>…</answer>` format.
* **KL penalty** keeps the policy close to a frozen copy of the SFT model (`π_ref`), so it doesn't forget language or "reward-hack" into gibberish.
* **Clipped ratio** (from PPO), `min(ρA, clip(ρ, 1±ε)A)`, so a single update can't move the policy too far.

**What RL practises.** Half of the RL questions are the **4–5-step stories that SFT never showed**. For those, nobody ever wrote a worked solution. The model has to extend the procedure it learned on shorter stories, and GRPO reinforces the attempts that reach the right number. The other half are the familiar tasks, which keeps those skills sharp.

In [ ]:
def reward_fn(completion, p):
    r = 0.0
    if well_formatted(completion, "think"): r += 0.1
    if parse_answer(completion) == p["answer"]: r += 1.0
    return r

def sequence_logprobs(model, full, prompt_len):
    """log π(token) for every completion token. full: (G, L) ids."""
    with autocast():
        logits, _ = model(full)
    logits = logits[:, prompt_len - 1:-1].float()                  # position t predicts token t+1
    return torch.log_softmax(logits, -1).gather(-1, full[:, prompt_len:].unsqueeze(-1)).squeeze(-1)

def completion_mask(completion_ids):
    """1 for tokens up to and including the first <|endoftext|>, 0 after."""
    is_eot = (completion_ids == EOT_ID).int()
    after_first = (torch.cumsum(is_eot, 1) - is_eot) > 0
    return (~after_first).float()

ref_model = copy.deepcopy(model).eval()                           # frozen π_ref = SFT model
for p_ in ref_model.parameters(): p_.requires_grad_(False)

In [ ]:
G            = 8                      # samples per question (the "group")
Q_PER_STEP   = 8 if FAST else 4       # questions per RL step
RL_STEPS     = 200 if FAST else 30
RL_LR        = 1e-5 if FAST else 3e-5
BETA         = 0.04                   # KL weight
CLIP_EPS     = 0.2
PPO_EPOCHS   = 2                      # reuse each batch of rollouts for 2 gradient steps
TEMPERATURE  = 1.0
RL_TASK_WEIGHTS = [0.25, 0.25, 0.5]  # arithmetic, letters, hard 4-5 step arithmetic (no worked solutions!)

opt = make_optimizer(model, RL_LR, wd=0.0)
rl_rng = random.Random(2024)
log = collections.defaultdict(list)
t0 = time.time()

for step in range(RL_STEPS):
    # ---------------- 1) ROLLOUTS: sample G completions for each question ----------------
    model.eval()
    batch = []
    for _ in range(Q_PER_STEP):
        p = make_problem(rl_rng, rl_rng.choices(list(ALL_TASKS), weights=RL_TASK_WEIGHTS)[0])
        prompt_ids = tok.encode(prompt_text(p, "think"))
        P = len(prompt_ids)
        with torch.no_grad():
            full = generate(model, torch.tensor([prompt_ids] * G, device=device), MAX_NEW, temperature=TEMPERATURE)
        comp_ids = full[:, P:]
        texts = [tok.decode(r.tolist()).split(EOT)[0] for r in comp_ids]
        rewards = torch.tensor([reward_fn(t, p) for t in texts], device=device)
        # ---------------- 2) GROUP-RELATIVE ADVANTAGE ----------------
        adv = (rewards - rewards.mean()) / (rewards.std() + 1e-4)
        mask = completion_mask(comp_ids)
        with torch.no_grad():
            old_lp = sequence_logprobs(model, full, P)
            ref_lp = sequence_logprobs(ref_model, full, P)
        batch.append(dict(full=full, P=P, adv=adv, mask=mask, old_lp=old_lp, ref_lp=ref_lp))
        log["reward"].append(rewards.mean().item())
        log["acc"].append((rewards >= 1.0).float().mean().item())
        log["length"].append(mask.sum(1).mean().item())

    # ---------------- 3) POLICY UPDATE (clipped surrogate + KL) ----------------
    # (we stay in eval mode = dropout off, so the new/old log-prob ratio starts at exactly 1)
    for _ in range(PPO_EPOCHS):
        opt.zero_grad(set_to_none=True)
        total_kl = 0.0
        for b in batch:
            if b["adv"].abs().sum() == 0:            # all samples equally good → no signal
                continue
            lp = sequence_logprobs(model, b["full"], b["P"])
            ratio = torch.exp(lp - b["old_lp"])
            A = b["adv"].unsqueeze(1)
            surrogate = torch.min(ratio * A, torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * A)
            # k3 KL estimator (always ≥ 0):  e^(ref−lp) − (ref−lp) − 1
            d = b["ref_lp"] - lp
            kl = torch.exp(d) - d - 1
            per_token = -(surrogate - BETA * kl)
            loss = ((per_token * b["mask"]).sum(1) / b["mask"].sum(1).clamp(min=1)).mean() / len(batch)
            loss.backward()
            total_kl += ((kl * b["mask"]).sum() / b["mask"].sum()).item() / len(batch)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    log["kl"].append(total_kl)

    if step % (20 if FAST else 5) == 0 or step == RL_STEPS - 1:
        k = Q_PER_STEP * (20 if FAST else 5)
        print(f"[GRPO] step {step:4d}/{RL_STEPS} | reward {sum(log['reward'][-k:])/len(log['reward'][-k:]):.3f} | "
              f"train acc {sum(log['acc'][-k:])/len(log['acc'][-k:]):.0%} | "
              f"len {sum(log['length'][-k:])/len(log['length'][-k:]):5.1f} | KL {total_kl:.4f} | {time.time()-t0:5.0f}s")

torch.save(model.state_dict(), WORK_DIR / "grpo.pt")

In [ ]:
def smooth(xs, k=None):
    k = k or max(1, len(xs) // 20)
    return [sum(xs[max(0, i - k):i + 1]) / len(xs[max(0, i - k):i + 1]) for i in range(len(xs))]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].plot(smooth(log["reward"])); axes[0].set_title("mean reward (sampled, T=1)")
axes[1].plot(smooth(log["length"])); axes[1].set_title("completion length (tokens)")
axes[2].plot(smooth(log["kl"])); axes[2].set_title("KL(π ‖ π_ref)")
for ax in axes: ax.set_xlabel("rollout group" if ax is not axes[2] else "RL step"); ax.grid(alpha=.3)
plt.suptitle("Stage 3: GRPO training"); plt.tight_layout(); plt.show()

## 10. Compare all stages 📊

Same held-out test problems, greedy decoding. Watch the **`arith_4-5_steps`** column: SFT never saw problems that long, so any gain there comes from RL practice.

In [ ]:
results["GRPO · think"] = evaluate(model, test_set, "think")
base = GPT(cfg).to(device); base.load_state_dict(torch.load(WORK_DIR / "base.pt", map_location=device))
results["Base · think"] = evaluate(base, test_set, "think"); del base

order = ["Base · think", "SFT · direct", "SFT · think", "GRPO · think"]
cats = list(ALL_TASKS) + ["all"]
fig, ax = plt.subplots(figsize=(9, 4))
w = 0.8 / len(order)
for i, name in enumerate(order):
    vals = [results[name].get(c, 0) for c in cats]
    bars = ax.bar([j + i * w for j in range(len(cats))], vals, w, label=name)
    ax.bar_label(bars, [f"{v:.0%}" for v in vals], fontsize=8)
ax.set_xticks([j + w * (len(order) - 1) / 2 for j in range(len(cats))], cats)
ax.set_ylim(0, 1.1); ax.set_ylabel("accuracy"); ax.legend(ncol=4, fontsize=8, loc="upper left")
ax.set_title("Accuracy on held-out problems"); plt.show()

for name in order:
    print(f"{name:14s}", {k: f"{v:.0%}" for k, v in results[name].items()})

## 11. Test-time compute: think more, and vote 🗳️

Reasoning models have a second knob besides training: **spend more compute when answering**. The simplest version is **self-consistency**: sample `N` independent chains of thought at temperature > 0, then take a **majority vote** over the final answers.

```
            ┌─ chain 1 → 12 ┐
 question ──┼─ chain 2 → 12 ├──▶ vote: 12 (2 votes) vs 11 (1 vote) ──▶ 12
            └─ chain 3 → 11 ┘
```

The more samples you draw, the more of the model's occasional slips get outvoted.

In [ ]:
N_MAX = 16 if FAST else 8
sc_set = test_set[::2]
samples = run_problems(model, sc_set, "think", temperature=0.7, n_samples=N_MAX, batch=128 if FAST else 32)

def majority_acc(n):
    correct = 0
    for p, outs in zip(sc_set, samples):
        answers = [a for a in (parse_answer(c) for c in outs[:n]) if a is not None]
        vote = collections.Counter(answers).most_common(1)[0][0] if answers else None
        correct += vote == p["answer"]
    return correct / len(sc_set)

ns = [n for n in [1, 2, 4, 8, 16] if n <= N_MAX]
accs = [majority_acc(n) for n in ns]
plt.figure(figsize=(5, 3.2)); plt.plot(ns, accs, "o-")
plt.xscale("log", base=2); plt.xticks(ns, ns); plt.xlabel("samples per question (N)"); plt.ylabel("accuracy")
plt.title("Self-consistency: accuracy vs test-time compute"); plt.grid(alpha=.3); plt.show()
print({n: f"{a:.0%}" for n, a in zip(ns, accs)})

## 12. Try it yourself 💬

Write your own question in the same style as the training problems, using names and objects from your books. The model only knows these two kinds of problems, so a question like "what is the capital of France?" won't work: **its reasoning is narrow because its training was narrow.**

In [ ]:
def ask(question, mode="think", temperature=0.0):
    p = {"question": question}
    x = torch.tensor([tok.encode(prompt_text(p, mode))], device=device)
    out = generate(model, x, MAX_NEW, temperature=temperature)
    print(tok.decode(out[0, x.shape[1]:].tolist()).split(EOT)[0])

a, b, obj = NAMES[0], NAMES[1], OBJECTS[0]
ask(f"{a} has 9 {obj}. {b} gives {a} 6 more {obj}. {a} loses 4 {obj}. How many {obj} does {a} have now?")
print()
ask(f"How many times does the letter 'r' appear in 'strawberry'?")

## 13. Save 💾

In [ ]:
torch.save({"model": model.state_dict(), "cfg": cfg, "results": results}, WORK_DIR / "reasoning_gpt.pt")
print("saved to", WORK_DIR.resolve())
if IN_COLAB and USE_GOOGLE_DRIVE:
    import shutil
    dst = Path("/content/drive/MyDrive/llm_from_scratch"); dst.mkdir(parents=True, exist_ok=True)
    for f in ["reasoning_gpt.pt", "tokenizer.json"]: shutil.copy(WORK_DIR / f, dst)
    print("copied to", dst)

## 🎉 Recap

```
 books ──pretrain──▶ base LM ──SFT on worked solutions──▶ follows <think>/<answer> ──GRPO (reward = correct?)──▶ reasoning model
                                                                                                                  │
                                                                              sample N chains + majority vote ◀──┘  (test-time compute)
```

| Stage | Signal | What it teaches |
|---|---|---|
| Pretraining | next token of the books | language, names, world of the books |
| SFT (cold start) | next token of *worked solutions* | the thinking format and a procedure |
| GRPO | only "was the final answer right?" | *reliability* and *generalization*: reinforce chains of thought that work, including on longer problems SFT never showed |
| Self-consistency | none (inference only) | trade compute for accuracy |

### Going further

1. **Harder problems**: 6+ steps, bigger numbers, multiplication, comparisons between two characters. Watch whether the completion length grows during RL.
2. **R1-Zero style**: skip SFT and run GRPO straight from the base model using only the format and answer reward. (This needs a bigger model to get off the ground.)
3. **Process rewards**: also check each `a + b = c` line inside `<think>`, not just the final answer.
4. **Scale**: a bigger model, more pretraining text, a KV-cache for faster rollouts.

**References:** DeepSeek-AI, *DeepSeek-R1* (2025) · Shao et al., *DeepSeekMath / GRPO* (2024) · Wei et al., *Chain-of-Thought Prompting* (2022) · Wang et al., *Self-Consistency* (2022) · Schulman et al., *PPO* (2017).